In [1]:
from load_data import domain_var, load_data, temperature_var, velocity_var

import qewton

Load data

In [ ]:
batch_size = qewton.optim.DiscreteHyperparameter((5, 50))
dataset, data_loader = load_data(batch_size)

In [3]:
print(domain_var)
print(temperature_var)
print(velocity_var)
print(len(dataset))

Variable(domain, dim=1)
Variable(temperature, dim=1)
Variable(velocity, dim=3, children=3)
581


Setup computation graph

In [4]:
graph = qewton.Graph()
model = qewton.algorithms.PCANet(
    input_variable=domain_var,
    output_variable=temperature_var * velocity_var,
    pca_n_input=32,
    pca_n_output=32,
    data_source_node=data_loader,
    fcn_hidden_layers=2,
    fcn_hidden_neurons=50,
)
concat = qewton.algorithms.building_blocks.ConcatVariables(
    [temperature_var, velocity_var],
    name="Concat temperature and velocity data",
)
split = qewton.algorithms.building_blocks.SplitVariables(
    [temperature_var, velocity_var],
    name="Split temperature and velocity prediction",
)
temperature_constraint = qewton.constraints.MSEConstraint(name="Temperature MSE")
velocity_constraint = qewton.constraints.MSEConstraint(name="Velocity MSE", weight=1/3) # higher dimension than temperature

# input features
graph.connect(
    data_loader.get_output_port(domain_var), model.input_ports[0]
)
# output features
graph.connect(
    data_loader.get_output_port(temperature_var),
    concat.get_input_port("temperature"),
)
graph.connect(
    data_loader.get_output_port(velocity_var),
    concat.get_input_port("velocity"),
)
graph.connect(
    concat.output_ports[0], model.input_ports[1]
)
# constraints
graph.connect(
    model, split.input_ports[0]
)
graph.connect(
    split.get_output_port(temperature_var), temperature_constraint.input_1
)
graph.connect(
    data_loader.get_output_port(temperature_var), temperature_constraint.input_2
)
graph.connect(
    split.get_output_port(velocity_var), velocity_constraint.input_1
)
graph.connect(
    data_loader.get_output_port(velocity_var), velocity_constraint.input_2
)


Setup trainer

In [5]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=2500,
)
trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase],
    graphs=[graph],
    training_objectives=[temperature_constraint, velocity_constraint],
    device="cuda:0"
)


Run the trainer

In [6]:
trainer.run()

Optimization Phase 1: 100%|██████████| 2500/2500 [01:07<00:00, 37.21it/s, loss=0.0789]
